In [6]:
import pandas as pd
from pathlib import Path
import cv2
import numpy as np
import matplotlib.image as mpimg
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [3]:
df = pd.read_csv('voice.csv') 
le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])   # female=0, male=1
print(dict(zip(le.classes_, le.transform(le.classes_))))

{'female': np.int64(0), 'male': np.int64(1)}


In [4]:
X = df.drop(columns='label')
y = df['label']

In [5]:
rows = []
for name, test_size in [('70:30', 0.30), ('80:20', 0.20)]:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=test_size, random_state=42, stratify=y)
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_te_s = sc.transform(X_te)
    for kernel in ['linear', 'poly', 'rbf']:
        m = SVC(kernel=kernel)
        m.fit(X_tr_s, y_tr)
        rows.append((name, kernel,
                     accuracy_score(y_tr, m.predict(X_tr_s)),
                     accuracy_score(y_te, m.predict(X_te_s))))
res = pd.DataFrame(rows, columns=['Split', 'Kernel', 'Acc_Train', 'Acc_Test'])
print(res.round(4).to_string(index=False))
print(res.pivot(index='Kernel', columns='Split', values='Acc_Test').round(4))

Split Kernel  Acc_Train  Acc_Test
70:30 linear     0.9720    0.9790
70:30   poly     0.9666    0.9600
70:30    rbf     0.9838    0.9832
80:20 linear     0.9767    0.9748
80:20   poly     0.9680    0.9558
80:20    rbf     0.9838    0.9826
Split    70:30   80:20
Kernel                
linear  0.9790  0.9748
poly    0.9600  0.9558
rbf     0.9832  0.9826


In [ ]:
train_dir = "images/training/"
test_dir = "images/test/"
 
def load_dataset(img_dir):
    p = Path(img_dir)
    img_list = []
    for d in p.glob('*'):
        label = d.name                      # 'day' / 'night'
        for file in d.glob('*.jpg'):
            img = mpimg.imread(file)
            if img is not None:
                img_list.append((img, label))
    return img_list
 
all_img = load_dataset(train_dir) + load_dataset(test_dir)
print(f'Total images: {len(all_img)}')